In [ ]:
import sys
import os
import subprocess
import shutil

# Check if we are running in Google Colab (remote Linux VM)
if 'google.colab' in sys.modules:
    print("Detected Google Colab environment. Setting up via GitHub...")

    # 1. Install required packages
    subprocess.run(["pip", "install", "-q", "torch_geometric", "prody"], check=True)

    # 2. Clone/update main repository
    repo_url = "https://github.com/WSobo/Struct2Seq-GCN.git"
    target_dir = "/content/Struct2Seq-GCN"

    if not os.path.exists(target_dir):
        os.chdir('/content')
        print("Cloning repository...")
        subprocess.run(["git", "clone", repo_url], check=True)
    else:
        os.chdir(target_dir)
        print("Pulling latest changes...")
        subprocess.run(["git", "pull"], check=True)

    # 3. Ensure LigandMPNN dependency exists (fallback for gitlink/submodule issues)
    ligand_dir = os.path.join(target_dir, "LigandMPNN")
    ligand_data_utils = os.path.join(ligand_dir, "data_utils.py")
    if not os.path.exists(ligand_data_utils):
        print("Fetching LigandMPNN dependency...")
        if os.path.exists(ligand_dir):
            shutil.rmtree(ligand_dir)
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/dauparas/LigandMPNN.git", "LigandMPNN"],
            cwd=target_dir,
            check=True,
        )

    # 4. Enter the project directory
    os.chdir(target_dir)
    print("Colab setup complete. Current Dir:", os.getcwd())

Detected Google Colab environment. Setting up via GitHub...
Pulling latest changes...
Updating submodules...


CalledProcessError: Command '['git', 'submodule', 'update', '--init', '--recursive']' returned non-zero exit status 128.

# Struct2Seq-GCN: Pipeline Testing
This notebook runs the basic sanity checks for the inference and training pipelines of the Struct2Seq-GCN model.

In [15]:
import os
# Ensure we operate from the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print("Current Working Directory:", os.getcwd())

Current Working Directory: /content/Struct2Seq-GCN


## Step 1: Test Inference Pipeline (Sanity Check)
Before training, make sure the model can successfully take a 3D structure, convert it to a graph, run it through the GCN, and output sequence predictions.

In [ ]:
import os

# Set PYTHONPATH environment variable so python subprocesses can find project modules
os.environ['PYTHONPATH'] = f"{os.environ.get('PYTHONPATH', '')}:{os.getcwd()}"

# Use an actual PDB file as input for parse_PDB
!python scripts/run.py --pdb LigandMPNN/inputs/1BC8.pdb

Traceback (most recent call last):
  File "/content/Struct2Seq-GCN/utils/graph_builder.py", line 11, in <module>
    from data_utils import parse_PDB, featurize
ModuleNotFoundError: No module named 'data_utils'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/Struct2Seq-GCN/scripts/run.py", line 7, in <module>
    from utils.graph_builder import pdb_to_pyg_data
  File "/content/Struct2Seq-GCN/utils/graph_builder.py", line 15, in <module>
    from data_utils import parse_PDB, featurize
ModuleNotFoundError: No module named 'data_utils'


## Step 2: Test Training Loop
Confirm the data loader can successfully read the LigandMPNN `train.json` splits and start optimizing the weights for a couple of epochs.

In [ ]:
import os
import json
import glob

# Build a tiny local split from available input PDBs for a stable smoke test
pdb_files = sorted(glob.glob("LigandMPNN/inputs/*.pdb"))
ids = [os.path.splitext(os.path.basename(p))[0] for p in pdb_files]
if len(ids) < 2:
    raise RuntimeError("Need at least 2 PDB files in LigandMPNN/inputs for train/valid smoke test.")

os.makedirs("training", exist_ok=True)
with open("training/train_smoke.json", "w") as f:
    json.dump([ids[0]], f)
with open("training/valid_smoke.json", "w") as f:
    json.dump([ids[1]], f)

!python scripts/train.py \
    --json_train training/train_smoke.json \
    --json_valid training/valid_smoke.json \
    --pdb_dir LigandMPNN/inputs/ \
    --epochs 2 \
    --batch_size 1